<a href="https://colab.research.google.com/github/somyadev725-create/crop-disease-classifier/blob/main/skillgap_matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import sklearn
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

pandas: 2.2.3
scikit-learn: 1.6.1


In [2]:
import pandas as pd

roles = pd.DataFrame({
    "role": [
        "Data Analyst",
        "Machine Learning Intern",
        "UI/UX Designer",
        "Python Developer",
        "Web Developer",
        "AI Research Intern",
    ],
    "required_skills": [
        "python, sql, excel, pandas, data visualization, statistics",
        "python, machine learning, numpy, pandas, scikit-learn, statistics, linear algebra",
        "figma, wireframing, user research, prototyping, ui design",
        "python, data structures, git, sql, apis, problem solving",
        "html, css, javascript, react, git, responsive design",
        "python, machine learning, deep learning, linear algebra, probability, research papers, pytorch",
    ],
})
roles

,role,required_skills
0,Data Analyst,"python, sql, excel, pandas, data visualization..."
1,Machine Learning Intern,"python, machine learning, numpy, pandas, sciki..."
2,UI/UX Designer,"figma, wireframing, user research, prototyping..."
3,Python Developer,"python, data structures, git, sql, apis, probl..."
4,Web Developer,"html, css, javascript, react, git, responsive ..."
5,AI Research Intern,"python, machine learning, deep learning, linea..."


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def split_skills(text):
    return [s.strip().lower() for s in text.split(",") if s.strip()]

vectorizer = TfidfVectorizer(tokenizer=split_skills, lowercase=False, token_pattern=None)
vectorizer.fit(roles["required_skills"])

def match(student_skills, target_role):
    row = roles[roles["role"].str.lower() == target_role.lower()]
    if row.empty:
        print("Role not found. Choose from:", list(roles["role"]))
        return
    required_text = row.iloc[0]["required_skills"]
    vectors = vectorizer.transform([student_skills.lower(), required_text])
    score = cosine_similarity(vectors[0], vectors[1])[0][0]
    have = split_skills(student_skills)
    missing = [s for s in split_skills(required_text) if s not in have]
    print(f"Role: {row.iloc[0]['role']}")
    print(f"Match score: {score * 100:.0f}%")
    print("Missing skills:", ", ".join(missing) if missing else "None")

match("Python, SQL, Linear Algebra", "Machine Learning Intern")

Role: Machine Learning Intern
Match score: 35%
Missing skills: machine learning, numpy, pandas, scikit-learn, statistics


In [4]:
def best_roles(student_skills, top=3):
    results = []
    for _, r in roles.iterrows():
        vectors = vectorizer.transform([student_skills.lower(), r["required_skills"]])
        results.append((r["role"], cosine_similarity(vectors[0], vectors[1])[0][0]))
    results.sort(key=lambda x: x[1], reverse=True)
    for role, score in results[:top]:
        print(f"{role}: {score * 100:.0f}%")

best_roles("python, matlab, linear algebra, probability, statistics, wireframing")

AI Research Intern: 44%
Machine Learning Intern: 39%
Data Analyst: 25%


In [5]:
new_roles = pd.DataFrame({
    "role": ["Data Scientist", "Cloud Engineer", "Cybersecurity Intern", "Business Analyst", "Software Engineer Intern", "NLP Intern"],
    "required_skills": [
        "python, statistics, machine learning, pandas, sql, data visualization, probability",
        "cloud computing, linux, networking, python, docker, git",
        "networking, linux, cybersecurity basics, python, problem solving, risk analysis",
        "excel, sql, data visualization, communication, requirement analysis, presentation",
        "data structures, algorithms, python, git, problem solving, oop",
        "python, nlp, machine learning, deep learning, text processing, linear algebra",
    ],
})
roles = pd.concat([roles, new_roles], ignore_index=True)
vectorizer.fit(roles["required_skills"])
print(len(roles))
best_roles("python, matlab, linear algebra, probability, statistics, wireframing", top=3)

12
Data Scientist: 45%
AI Research Intern: 40%
Machine Learning Intern: 37%
